# Bangla Sign Language preprocessing

Clean notebook for converting `.webm` sign-language videos into MediaPipe keypoint `.npy` feature files and preparing Bangla text token IDs.

Run cells from top to bottom. Change only the path constants if your local folders differ.

## 1. Imports and configuration

In [1]:
import os
import json
from collections import Counter

import cv2
import numpy as np
pandas_imported = False
import pandas as pd
import mediapipe as mp
from tqdm import tqdm

print("pandas:", pd.__version__)
print("cv2:", cv2.__version__)
print("numpy:", np.__version__)
print("mediapipe:", mp.__version__)
print("mediapipe has solutions:", "solutions" in dir(mp))

pandas: 2.3.3
cv2: 4.11.0
numpy: 1.26.4
mediapipe: 0.10.14
mediapipe has solutions: True


In [ ]:
# ==== EDIT THESE PATHS IF NEEDED ====
ROOT = "/Volumes/Easha/SSD Backup/Bornil_Dataset/archive"
CSV_PATH = os.path.join(ROOT, "video_data.csv")

# Features are regenerated in a fresh "v2" folder because the frame handling
# changed: clips are now sampled UNIFORMLY across the whole video up to
# MAX_FRAMES instead of being truncated to the first MAX_FRAMES. The old
# first-100-frame files in features_mediapipe are incompatible, so keeping a
# separate folder gives a clean, reproducible set (delete the old one when ready).
FEATURE_DIR = "/Volumes/Easha/SSD Backup/Bornil_Dataset/features_mediapipe_v2"

# Cap on frames per clip. Videos longer than this are sampled uniformly across
# the whole clip (see sample_frame_indices); shorter videos are kept in full.
# Median clip is ~153 frames and 78% exceed 100, so 128 covers most signs
# end-to-end while bounding sequence length.
MAX_FRAMES = 128
PROCESS_LIMIT = 200

os.makedirs(FEATURE_DIR, exist_ok=True)

print("CSV exists:", os.path.exists(CSV_PATH))
print("Feature dir:", FEATURE_DIR)

## 2. Load CSV and verify video paths

In [3]:
df = pd.read_csv(CSV_PATH)

print("Rows:", len(df))
print("Columns:", list(df.columns))
df.head(3)

Rows: 17988
Columns: ['text', 'recording', 'metadata', 'frame_count', 'chunk_path']


,text,recording,metadata,frame_count,chunk_path
0,রাষ্ট্রদূত শাহের ফিলিস্তিনিদের প্রতি সহমর্মিতা...,f07dbede-8ed9-4754-aa68-683824f10e16.webm,"{""view"": ""front"", ""end_at"": 24.455, ""duration""...",732,chunks/chunk_1
1,এই আমটি একদমই মিষ্টি না,01c66808-0645-4139-bd38-4a09de210bf7.webm,"{""view"": ""front"", ""end_at"": 9.944, ""duration"":...",142,chunks/chunk_1
2,সংক্ষিপ্ত স্কোর শ্রীলঙ্কা ওভারে থিরিমান্নে দিল...,28346709-d0df-4006-a20c-0bd0c0163724.webm,"{""view"": ""front"", ""end_at"": 21.949, ""duration""...",658,chunks/chunk_1


In [4]:
def get_video_path(row):
    return os.path.join(ROOT, row["chunk_path"], row["recording"])

def get_sample(i):
    row = df.iloc[i]
    video_path = get_video_path(row)
    text = row["text"]
    return video_path, text

vp, txt = get_sample(0)
print(vp)
print("Exists:", os.path.exists(vp))
print(txt)

/Volumes/T7/Bornil_Dataset/archive/chunks/chunk_1/f07dbede-8ed9-4754-aa68-683824f10e16.webm
Exists: True
রাষ্ট্রদূত শাহের ফিলিস্তিনিদের প্রতি সহমর্মিতা প্রকাশ করার জন্য খালেদা জিয়ার কাছে কৃতজ্ঞতা প্রকাশ করেন


In [5]:
# Quick video sanity check
cap = cv2.VideoCapture(vp)
print("Opened:", cap.isOpened())
print("Frame count:", int(cap.get(cv2.CAP_PROP_FRAME_COUNT)))
print("FPS:", cap.get(cv2.CAP_PROP_FPS))
cap.release()

Opened: True
Frame count: 732
FPS: 29.916666666666668


## 3. MediaPipe keypoint extraction

Output per frame has 258 values:
- Pose: 33 landmarks × 4 values = 132
- Left hand: 21 landmarks × 3 values = 63
- Right hand: 21 landmarks × 3 values = 63

Frames are sampled **uniformly across the whole clip** up to `MAX_FRAMES`
(clips shorter than that are kept in full), so long videos are represented
end-to-end instead of being truncated to the first few seconds.

In [ ]:
# Keypoint extraction now lives in bsl_extract.py (single source of truth, also
# used by run_extraction.py for parallel full-dataset extraction). Output per
# frame is 258 raw values: pose 33x4 + left hand 21x3 + right hand 21x3,
# uniformly sampled to at most MAX_FRAMES across the whole clip.
from bsl_extract import (
    sample_frame_indices,
    keypoints_from_results,
    count_frames,
    extract_keypoints_from_video,
)

In [ ]:
# Test on one video
features = extract_keypoints_from_video(vp, max_frames=MAX_FRAMES)
print(features.shape)  # expected: (min(total_frames, MAX_FRAMES), 258)

## 4. Save features for many videos

In [ ]:
def get_feature_path(recording):
    return os.path.join(FEATURE_DIR, recording.replace(".webm", ".npy"))

def process_dataset(df, limit=200, max_frames=128):
    processed = 0
    skipped_existing = 0
    missing = 0
    failed = 0

    for i in tqdm(range(min(limit, len(df)))):
        row = df.iloc[i]
        video_path = get_video_path(row)
        save_path = get_feature_path(row["recording"])

        if os.path.exists(save_path):
            skipped_existing += 1
            continue

        if not os.path.exists(video_path):
            print("Missing:", video_path)
            missing += 1
            continue

        try:
            features = extract_keypoints_from_video(video_path, max_frames=max_frames)
            np.save(save_path, features)
            processed += 1
        except Exception as e:
            print("Failed:", video_path, "|", repr(e))
            failed += 1

    print("Processed new:", processed)
    print("Skipped existing:", skipped_existing)
    print("Missing:", missing)
    print("Failed:", failed)

In [8]:
# Process first 200 rows. Existing .npy files are skipped automatically.
process_dataset(df, limit=PROCESS_LIMIT, max_frames=MAX_FRAMES)

  0%|          | 0/200 [00:00<?, ?it/s]WARNING: All log messages before absl::InitializeLog() is called are written to STDERR
I0000 00:00:1777695839.056106 17910324 gl_context.cc:357] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M2
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1777695839.165305 17912375 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1777695839.177490 17912374 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1777695839.179975 17912376 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1777695839.180198 17912379 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling sup

Processed new: 180
Skipped existing: 20
Missing: 0
Failed: 0


In [9]:
# Verify saved files
files = [f for f in os.listdir(FEATURE_DIR) if f.endswith(".npy")]
print("Saved feature files:", len(files))

if files:
    sample = np.load(os.path.join(FEATURE_DIR, files[0]))
    print("Sample shape:", sample.shape)

Saved feature files: 200
Sample shape: (100, 258)


In [10]:
# Analyze feature lengths for processed files
lengths = []
for f in files:
    arr = np.load(os.path.join(FEATURE_DIR, f))
    lengths.append(arr.shape[0])

if lengths:
    print("Min frames:", min(lengths))
    print("Max frames:", max(lengths))
    print("Avg frames:", sum(lengths) / len(lengths))

Min frames: 57
Max frames: 100
Avg frames: 98.335


## 5. Bangla text cleaning and word-level tokenization

In [11]:
def clean_text(text):
    text = str(text).strip()
    text = " ".join(text.split())
    return text

df["clean_text"] = df["text"].apply(clean_text)

word_counter = Counter()
for sentence in df["clean_text"]:
    word_counter.update(sentence.split())

special_tokens = ["<pad>", "<bos>", "<eos>", "<unk>"]
word2idx = {tok: i for i, tok in enumerate(special_tokens)}

for word, count in word_counter.items():
    if word not in word2idx:
        word2idx[word] = len(word2idx)

idx2word = {i: w for w, i in word2idx.items()}

print("Vocabulary size:", len(word2idx))
print("Most common words:", word_counter.most_common(20))

Vocabulary size: 25547
Most common words: [('ও', 1175), ('করে', 981), ('না।', 926), ('আমি', 920), ('তিনি', 887), ('না', 828), ('এই', 753), ('এ', 678), ('আমার', 677), ('আর', 656), ('থেকে', 646), ('করা', 587), ('করতে', 550), ('সে', 547), ('তার', 544), ('জন্য', 496), ('আমাদের', 470), ('কথা', 466), ('হবে', 423), ('এক', 419)]


In [12]:
def encode_sentence(sentence):
    words = clean_text(sentence).split()
    ids = [word2idx["<bos>"]]
    ids += [word2idx.get(w, word2idx["<unk>"]) for w in words]
    ids.append(word2idx["<eos>"])
    return ids

def decode_tokens(token_ids):
    words = []
    for idx in token_ids:
        word = idx2word.get(int(idx), "<unk>")
        if word in ["<pad>", "<bos>", "<eos>"]:
            continue
        words.append(word)
    return " ".join(words)

tokens = encode_sentence(df.iloc[0]["text"])
print(df.iloc[0]["text"])
print(tokens)
print(decode_tokens(tokens))

রাষ্ট্রদূত শাহের ফিলিস্তিনিদের প্রতি সহমর্মিতা প্রকাশ করার জন্য খালেদা জিয়ার কাছে কৃতজ্ঞতা প্রকাশ করেন
[1, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 9, 16, 2]
রাষ্ট্রদূত শাহের ফিলিস্তিনিদের প্রতি সহমর্মিতা প্রকাশ করার জন্য খালেদা জিয়ার কাছে কৃতজ্ঞতা প্রকাশ করেন


## 6. Create training dataframe for processed samples

In [ ]:
df["feature_path"] = df["recording"].apply(get_feature_path)
df["token_ids"] = df["clean_text"].apply(encode_sentence)

df_processed = df[df["feature_path"].apply(os.path.exists)].copy().reset_index(drop=True)

print("Processed samples available for modeling:", len(df_processed))
df_processed[["text", "recording", "feature_path", "token_ids"]].head()

In [15]:
# Build processed dataframe: only rows whose .npy feature file exists

def get_feature_path(recording):
    return os.path.join(FEATURE_DIR, recording.replace(".webm", ".npy"))

df["feature_path"] = df["recording"].apply(get_feature_path)

df_processed = df[df["feature_path"].apply(os.path.exists)].copy()

print("Processed rows:", len(df_processed))
print(df_processed[["text", "recording", "chunk_path", "feature_path"]].head())

Processed rows: 200
                                                text  \
0  রাষ্ট্রদূত শাহের ফিলিস্তিনিদের প্রতি সহমর্মিতা...   
1                            এই আমটি একদমই মিষ্টি না   
2  সংক্ষিপ্ত স্কোর শ্রীলঙ্কা ওভারে থিরিমান্নে দিল...   
3  ঘ ও প্রাণী দুটিকে ভিন্ন ভিন্ন পর্বে রাখার যৌক্...   
4                        আমার স্বামী একজন ইঞ্জিনিয়ার   

                                   recording      chunk_path  \
0  f07dbede-8ed9-4754-aa68-683824f10e16.webm  chunks/chunk_1   
1  01c66808-0645-4139-bd38-4a09de210bf7.webm  chunks/chunk_1   
2  28346709-d0df-4006-a20c-0bd0c0163724.webm  chunks/chunk_1   
3  12228a08-cabd-47b7-b8ec-b21732e5aa8f.webm  chunks/chunk_1   
4  c68e3f1a-617f-4f6c-a9df-12865acacd10.webm  chunks/chunk_1   

                                        feature_path  
0  /Volumes/T7/Bornil_Dataset/features_mediapipe/...  
1  /Volumes/T7/Bornil_Dataset/features_mediapipe/...  
2  /Volumes/T7/Bornil_Dataset/features_mediapipe/...  
3  /Volumes/T7/Bornil_Dataset/features

In [16]:
# Save a lightweight manifest for modeling later
MANIFEST_PATH = os.path.join(ROOT, "processed_manifest.csv")
df_processed[["text", "clean_text", "recording", "chunk_path", "feature_path"]].to_csv(MANIFEST_PATH, index=False)
print("Saved manifest:", MANIFEST_PATH)

Saved manifest: /Volumes/T7/Bornil_Dataset/archive/processed_manifest.csv


## Next step

Features are saved **raw**: all 33 pose landmarks + both hands, `(T, 258)`, no
normalization. Turn them into model input at load time with `bsl_utils.py`:

- `normalize_keypoints` — body-anchors each frame to the shoulders (translation
  + scale invariance), keeps the `(T, 258)` shape.
- `select_keypoints` — drops pose landmarks 23–32 (hips/legs/feet), which
  MediaPipe reports at visibility ≤ 0.08 and places below the frame (noise),
  giving `(T, 218)`.
- `prepare_features(x, normalize=True, upper_body=True)` — does both; use it
  everywhere features are loaded so train/val/inference stay identical.

Because the raw files are complete, the normalization and landmark choices can
change without re-running MediaPipe. Size the model with `feature_dim()`
(= 218) rather than a hard-coded number, so the input contract never drifts.

After ~200 processed samples, the next notebook should build the first baseline
model (small BiLSTM/CTC or a simple encoder-decoder), loading features through a
dataset that calls `prepare_features` — see `SignDataset` in `Untitled-1.ipynb`.